### Comparing Aggregate Models for Regression

This try-it focuses on utilizing ensemble models in a regression setting.  Much like you have used individual classification estimators to form an ensemble of estimators -- here your goal is to explore ensembles for regression models.  As with your earlier assignment, you will use scikitlearn to carry out the ensembles using the `VotingRegressor`.   


#### Dataset and Task

Below, a dataset containing census information on individuals and their hourly wage is loaded using the `fetch_openml` function.  OpenML is another repository for datasets [here](https://www.openml.org/).  Your task is to use ensemble methods to explore predicting the `wage` column of the data.  Your ensemble should at the very least consider the following models:

- `LinearRegression` -- perhaps you even want the `TransformedTargetRegressor` here.
- `KNeighborsRegressor`
- `DecisionTreeRegressor`
- `Ridge`
- `SVR`

Tune the `VotingRegressor` to try to optimize the prediction performance and determine if the wisdom of the crowd performed better in this setting than any of the individual models themselves.  Report back on your findings and discuss the interpretability of your findings.  Is there a way to determine what features mattered in predicting wages?

In [59]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.datasets import fetch_openml
from sklearn.ensemble import VotingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

# Load dataset
survey = fetch_openml(data_id=534, as_frame=True).frame

In [3]:
# Display the first few rows to understand the data
survey.head()

,EDUCATION,SOUTH,SEX,EXPERIENCE,UNION,WAGE,AGE,RACE,OCCUPATION,SECTOR,MARR
0,8,no,female,21,not_member,5.10,35,Hispanic,Other,Manufacturing,Married
1,9,no,female,42,not_member,4.95,57,White,Other,Manufacturing,Married
2,12,no,male,1,not_member,6.67,19,White,Other,Manufacturing,Unmarried
3,12,no,male,4,not_member,4.00,22,White,Other,Other,Unmarried
4,12,no,male,17,not_member,7.50,35,White,Other,Other,Married


In [9]:
X = survey.drop('WAGE', axis=1)
y = survey['WAGE']
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

In [22]:
model_results = {
    'LinearRegression': {
        'model': Pipeline([('hot', OneHotEncoder(handle_unknown='ignore')), ('scaler', StandardScaler(with_mean=False)),
                           ('lr', LinearRegression())]),
    },
    'Ridge': {
        'model': Pipeline([('hot', OneHotEncoder(handle_unknown='ignore')), ('scaler', StandardScaler(with_mean=False)),
                           ('ridge', Ridge())])
    },
    'KNeighborsRegressor': {
        'model': Pipeline([('hot', OneHotEncoder(handle_unknown='ignore')), ('scaler', StandardScaler(with_mean=False)),
                           ('knr', KNeighborsRegressor())])
    },
    'DecisionTreeRegressor': {
        'model': Pipeline([('hot', OneHotEncoder(handle_unknown='ignore')), ('dt', DecisionTreeRegressor())])
    },
    'SVR': {
        'model': Pipeline([('hot', OneHotEncoder(handle_unknown='ignore')), ('scaler', StandardScaler(with_mean=False)),
                           ('svr', SVR())])
    },
    'VotingRegressor': {
        'model': Pipeline([('hot', OneHotEncoder(handle_unknown='ignore')), ('scaler', StandardScaler(with_mean=False)),
                           ('voter', VotingRegressor(
                               estimators=[
                                   ('lr', LinearRegression()),
                                   ('ridge', Ridge()),
                                   ('knr', KNeighborsRegressor()),
                                   ('dt', DecisionTreeRegressor()),
                                   ('svr', SVR())
                               ]
                           ))])
    },
}

In [23]:
for k, v in model_results.items():
    print(f'executing model {k}')
    model = v['model']
    model.fit(X_train, y_train)
    v['mse'] = mean_squared_error(y_test, model.predict(X_test))


executing model LinearRegression
executing model Ridge
executing model KNeighborsRegressor
executing model DecisionTreeRegressor
executing model SVR
executing model VotingRegressor


In [54]:
df = pd.DataFrame(model_results)
df = df.drop('model')

In [56]:
df = df.T.reset_index()
df.columns = ['model', 'mse']
df


,model,mse
0,LinearRegression,30.502651
1,Ridge,26.390316
2,KNeighborsRegressor,25.727666
3,DecisionTreeRegressor,26.237224
4,SVR,24.108368
5,VotingRegressor,21.305941


In [57]:
fig = px.bar(df, x='model', y='mse', title="Ensemble Voter performed the best")
fig.show()
fig.write_image('images/compare.png')

In [60]:

# Extract fitted pipeline and components
pipe = model_results['VotingRegressor']['model']
enc = pipe.named_steps['hot']
voter = pipe.named_steps['voter']

# Feature names after OneHotEncoding
feature_names = enc.get_feature_names_out(input_features=X_train.columns)

# Collect normalized importances from estimators that support it
imps = []
for est in getattr(voter, 'estimators_', []):
    if hasattr(est, 'feature_importances_'):
        imp = np.asarray(est.feature_importances_, dtype=float)
    elif hasattr(est, 'coef_'):
        imp = np.asarray(est.coef_, dtype=float).ravel()
        imp = np.abs(imp)
    else:
        continue
    s = imp.sum()
    if s > 0 and imp.shape[0] == len(feature_names):
        imps.append(imp / s)

# If no importances available, create zeros to avoid errors
if len(imps) == 0:
    importance = np.zeros(len(feature_names))
else:
    importance = np.mean(np.vstack(imps), axis=0)

# Build DataFrame and plot with Plotly Express
feat_imp_df = pd.DataFrame({'feature': feature_names, 'importance': importance}).sort_values('importance',
                                                                                             ascending=False)
top_n = 30 if len(feat_imp_df) > 30 else len(feat_imp_df)

fig = px.bar(
    feat_imp_df.head(top_n),
    x='importance',
    y='feature',
    orientation='h',
    title='VotingRegressor Feature Importance Occupation and Experience most important'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()
fig.write_image('images/feature_importance.png')
feat_imp_df


,feature,importance
11,EDUCATION_14,0.033264
124,OCCUPATION_Management,0.033046
126,OCCUPATION_Professional,0.027487
21,EXPERIENCE_1,0.023776
72,UNION_not_member,0.023248
...,...,...
2,EDUCATION_5,0.001435
10,EDUCATION_13,0.001191
40,EXPERIENCE_20,0.001073
38,EXPERIENCE_18,0.001053
